In [0]:
-- 1. Create mapping table
CREATE OR REPLACE TABLE meridian_tnn.gov.store_entitlements (
    user_email STRING,
    allowed_state STRING,
    role_description STRING,
    created_at TIMESTAMP
);

-- 2. Seed entitlements (start with TN for testing)
INSERT INTO meridian_tnn.gov.store_entitlements VALUES
    ('analyst_tn@meridian.com', 'TN',  'Regional Analyst - Tennessee', current_timestamp()),
    ('analyst_ca@meridian.com', 'CA',  'Regional Analyst - California', current_timestamp()),
    ('auditor@meridian.com',    'ALL', 'Global Compliance Auditor',   current_timestamp()),
    (current_user(),            'TN',  'Active User - Single State',  current_timestamp());

-- 3. Unified row filter function (used by BOTH tables)
CREATE OR REPLACE FUNCTION meridian_tnn.gov.state_row_filter(state_code STRING)
RETURNS BOOLEAN
RETURN 
    EXISTS (
        SELECT 1 
        FROM meridian_tnn.gov.store_entitlements e
        WHERE e.user_email = current_user()
          AND (e.allowed_state = state_code OR e.allowed_state = 'ALL')
    );


-- 1. Attach directly to the Fact Table on 'store_state'
ALTER MATERIALIZED VIEW meridian_tnn.gold.fact_store_returns 
SET ROW FILTER meridian_tnn.gov.state_row_filter ON (store_state);

-- 2. Attach directly to the Store Dimension on 's_state'
ALTER MATERIALIZED VIEW meridian_tnn.gold.dim_store 
SET ROW FILTER meridian_tnn.gov.state_row_filter ON (s_state);


-- Test 1: Active user has 'TN' entitlement -> Should return TN stores and all fact records
SELECT DISTINCT s_state, COUNT(*) AS store_count 
FROM meridian_tnn.gold.dim_store 
GROUP BY s_state;

SELECT COUNT(*) AS total_returns 
FROM meridian_tnn.gold.fact_store_returns;

-- Test 2: Switch active user to 'CA' -> Should return 0 rows for both tables
UPDATE meridian_tnn.gov.store_entitlements 
SET allowed_state = 'CA' 
WHERE user_email = current_user();

SELECT COUNT(*) AS store_count 
FROM meridian_tnn.gold.dim_store;

SELECT COUNT(*) AS return_records 
FROM meridian_tnn.gold.fact_store_returns;

-- Test 3: Reset active user to 'ALL' -> Restores full global visibility
UPDATE meridian_tnn.gov.store_entitlements 
SET allowed_state = 'ALL' 
WHERE user_email = current_user();

SELECT COUNT(*) AS store_count 
FROM meridian_tnn.gold.dim_store;

SELECT COUNT(*) AS return_records 
FROM meridian_tnn.gold.fact_store_returns;


-- -- 1. Mask for Customer Surrogate Key (Two-Tier Masking)
-- CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_customer_id(customer_id INT)
-- RETURNS INT
-- RETURN 
--     CASE 
--         WHEN IS_ACCOUNT_GROUP_MEMBER('t08_pii_readers') THEN customer_id
--         ELSE -ABS(HASH(customer_id) % 1000000)
--     END;

-- -- 2. Mask for Financial Refunds (Redacted / Rounded)
-- CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_refund_cash(cash DECIMAL(10,2))
-- RETURNS DECIMAL(10,2)
-- RETURN 
--     CASE 
--         WHEN IS_ACCOUNT_GROUP_MEMBER('t08_pii_readers') THEN cash
--         ELSE ROUND(cash, -2)
--     END;

-- -- 3. Mask for Customer Name (Partial Masking)
-- CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_customer_name(val STRING)
-- RETURNS STRING
-- RETURN 
--     CASE 
--         WHEN IS_ACCOUNT_GROUP_MEMBER('t08_pii_readers') THEN val
--         WHEN val IS NOT NULL THEN CONCAT(SUBSTRING(val, 1, 1), '****')
--         ELSE NULL
--     END;

-- 1. Customer Name Mask
CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_customer_name(val STRING)
RETURNS STRING
RETURN 
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('p08') THEN val
        WHEN val IS NOT NULL THEN CONCAT(SUBSTRING(val, 1, 1), '****')
        ELSE NULL
    END;

-- 2. Customer ID Mask
CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_customer_id(customer_id INT)
RETURNS INT
RETURN 
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('p08') THEN customer_id
        ELSE -ABS(HASH(customer_id) % 1000000)
    END;

-- 3. Refund Amount Mask
CREATE OR REPLACE FUNCTION meridian_tnn.gov.mask_refund_cash(cash DECIMAL(10,2))
RETURNS DECIMAL(10,2)
RETURN 
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('p08') THEN cash
        ELSE ROUND(cash, -2)
    END;



-- Attach Column Masks
ALTER MATERIALIZED VIEW meridian_tnn.gold.fact_store_returns 
ALTER COLUMN customer_sk SET MASK meridian_tnn.gov.mask_customer_id;

ALTER MATERIALIZED VIEW meridian_tnn.gold.fact_store_returns 
ALTER COLUMN refunded_cash SET MASK meridian_tnn.gov.mask_refund_cash;

ALTER MATERIALIZED VIEW meridian_tnn.gold.dim_customer 
ALTER COLUMN c_first_name SET MASK meridian_tnn.gov.mask_customer_name;

ALTER MATERIALIZED VIEW meridian_tnn.gold.dim_customer 
ALTER COLUMN c_last_name SET MASK meridian_tnn.gov.mask_customer_name;



--testing---
-- Check Fact Table Masking
SELECT 
    ticket_number,
    customer_sk,
    refunded_cash
FROM meridian_tnn.gold.fact_store_returns
WHERE refunded_cash IS NOT NULL
LIMIT 5;

-- Check Customer Dimension Masking
SELECT 
    c_customer_sk,
    c_first_name,
    c_last_name
FROM meridian_tnn.gold.dim_customer
LIMIT 5;


SELECT 
    SUM(total_return_value) AS total_cash,
    SUM(total_units_returned) AS total_units
FROM meridian_tnn.gold.returns_metric_view;

-- Allow consumers to use the catalog and gold schema
GRANT USE CATALOG ON CATALOG meridian_tnn TO `account users`;
GRANT USE SCHEMA ON SCHEMA meridian_tnn.gold TO `account users`;
GRANT SELECT ON SCHEMA meridian_tnn.gold TO `account users`;

-- Explicitly revoke access to Bronze and Silver
REVOKE ALL PRIVILEGES ON SCHEMA meridian_tnn.bronze FROM `account users`;
REVOKE ALL PRIVILEGES ON SCHEMA meridian_tnn.silver FROM `account users`;



-- Audit Query: Proves every column tagged as sensitive has an active mask
WITH tagged_columns AS (
    SELECT 
        schema_name,
        table_name,
        column_name,
        tag_value AS sensitivity_level
    FROM meridian_tnn.information_schema.column_tags
    WHERE schema_name = 'gold'
      AND tag_name = 'data_sensitivity'
      AND tag_value IN ('Restricted_PII', 'Confidential')
),
masked_columns AS (
    SELECT 
        table_schema,
        table_name,
        column_name,
        mask_name
    FROM meridian_tnn.information_schema.column_masks
    WHERE table_schema = 'gold'
)
SELECT 
    t.schema_name,
    t.table_name,
    t.column_name,
    t.sensitivity_level,
    COALESCE(m.mask_name, 'FAIL - NO MASK') AS applied_mask,
    CASE 
        WHEN m.mask_name IS NOT NULL THEN 'COMPLIANT' 
        ELSE 'NON-COMPLIANT' 
    END AS compliance_status
FROM tagged_columns t
LEFT JOIN masked_columns m
    ON t.schema_name = m.table_schema
   AND t.table_name = m.table_name
   AND t.column_name = m.column_name
ORDER BY t.table_name, t.column_name;


-- Check Customer Dimension (Should show cleartext full names)
SELECT 
    c_customer_sk, 
    c_first_name, 
    c_last_name 
FROM meridian_tnn.gold.dim_customer 
ORDER BY rand() 
LIMIT 5;

-- Check Fact Table (Should show positive customer keys and exact decimals)
SELECT 
    ticket_number, 
    customer_sk, 
    refunded_cash 
FROM meridian_tnn.gold.fact_store_returns 
WHERE refunded_cash IS NOT NULL 
ORDER BY rand() 
LIMIT 5;